Historical preparation/training reproduction. The completed main test is closed. Use a separate work directory for any independent training; do not substitute its new weights into the frozen evaluated service. Private inputs and authorized W&B access are required.


# Real-Abshar Phase 2 — M5: edited-region segmentation

Attach the SAME private prepared-data input used for M4. Enable GPU + Internet and Files-only
persistence. Grant this notebook access to `WANDB_API_KEY`. Keep your M4 Models ZIP separately
for integration; it is not an input to this independent segmentation training.

The cohort remains **552 train / 126 validation / 114 test images** from **92 / 21 / 19** complete
source families. Test images are not used for model selection or evaluated here. Masks already
contain background 0 / edited-region 1. They mark edit/inpainting regions, which can be broader
than object silhouettes. Source annotations and source split assignments stay unchanged.

Compare an FCN-style ResNet18 baseline with a U-Net-style ResNet18 segmenter. Each uses ImageNet
initialization, input 640×640, batch 4, seed 42, FP32, up to 25 epochs and patience 6. Validation
chooses best checkpoints at threshold 0.50, then tunes thresholds 0.05…0.95 and selects the model.
Both initial models are evaluated before task fine-tuning at threshold 0.50 to measure what training adds.
M8 will run segmentation for every image, independently of the M4 detector's output.


In [ ]:
# 1. Retain Kaggle's PyTorch/torchvision GPU build; pin W&B for checkpoint backup.
import importlib.metadata
import os
import subprocess
import sys

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['WANDB_DISABLE_CODE'] = 'true'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'wandb==0.28.1'])
print({name: importlib.metadata.version(name) for name in ['torch', 'torchvision', 'wandb', 'numpy', 'Pillow']})


In [ ]:
# 2. Paths, confirmed Phase 2 W&B project, and secure API-key access.
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/real_abshar_phase2_m5_v1')
PREPARED_ROOT = None
WANDB_PROJECT = 'Real-Abshar-Phase2'
WANDB_ENTITY = 'f1k80-kaust'
RESTORE_ARTIFACT = None  # Future M5 restore only: 'entity/project/real-abshar-m5-backup-ID:vN'.

if not os.environ.get('WANDB_API_KEY'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    except Exception:
        raise RuntimeError('Enable this notebook access to the WANDB_API_KEY Kaggle secret before training.') from None
if not os.environ.get('WANDB_API_KEY'):
    raise RuntimeError('WANDB_API_KEY is empty. Fix the Kaggle secret before training.')
print('W&B key loaded securely; live artifact upload is checked before fitting.')


## 3. Embedded code and cohort check

The code is written into the output folder and its SHA-256 values are locked. No extra code
dataset upload is needed. An optional own-project M5 artifact can be restored if specified.
The data check must show the same 552/126/114 cohort as M4; no source is reassigned.


In [ ]:
# 3. Embedded sources; preserve these files when resuming a locked run.
SOURCES = {'cohort_data.py': '"""Load the reviewed preparation and preserve a complete-family Phase 2 cohort."""\nfrom pathlib import Path, PurePosixPath\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nimport shutil\nimport stat\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nSOURCE_MANIFEST_SHA256 = "8e964394838aba82aa85d8e7b93551a4018f0c1d7ce1e96ade17b625aeee14b8"\nSOURCE_ASSIGNMENT_SHA256 = "bcef2a4e57004f1e66645a9ea8a1d14275c3d8f0c7bbe45625760513623ab7cd"\nCOHORT_UID_SHA256 = "c24eb30ac2269840eb4f92e5ada2870493e7624c211a96cb8791d80ce8f40476"\nCOHORT_NAME = "csi-complete-aligned-families-v1"\nEXPECTED = {"train": {"images": 552, "edited": 460, "pristine": 92, "sources": 92},\n            "val": {"images": 126, "edited": 105, "pristine": 21, "sources": 21},\n            "test": {"images": 114, "edited": 95, "pristine": 19, "sources": 19}}\n\n\ndef sha_file(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef dump(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef write_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fields = fields or list(rows[0])\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_path(root, relative):\n    value = str(relative)\n    p = PurePosixPath(value)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in value or ":" in value:\n        raise ValueError("Unsafe dataset-relative path: " + value)\n    candidate = (Path(root) / value).resolve()\n    if not candidate.is_relative_to(Path(root).resolve()):\n        raise ValueError("Dataset path escapes its root.")\n    return candidate\n\n\ndef filter_complete_families(rows):\n    families = defaultdict(list)\n    for row in rows:\n        families[row["source_id"]].append(row)\n    complete = set()\n    for source, values in families.items():\n        if (len(values) == 6 and {int(r["variant"]) for r in values} == set(range(6))\n                and sum(r["kind"] == "pristine" for r in values) == 1\n                and sum(r["kind"] == "edited" for r in values) == 5):\n            complete.add(source)\n    kept = [row for row in rows if row["source_id"] in complete]\n    omitted = [{"uid": row["uid"], "source_id": row["source_id"], "split": row["split"],\n                "reason": "incomplete_source_family_after_annotation_QC"}\n               for row in rows if row["source_id"] not in complete]\n    return kept, omitted\n\n\ndef split_counts(rows):\n    result = {}\n    for split in ["train", "val", "test"]:\n        part = [r for r in rows if r["split"] == split]\n        result[split] = {"images": len(part), "edited": sum(r["kind"] == "edited" for r in part),\n                         "pristine": sum(r["kind"] == "pristine" for r in part),\n                         "sources": len({r["source_id"] for r in part})}\n    return result\n\n\ndef read_preparation(root):\n    root = Path(root)\n    manifest = root / "manifest.csv"\n    if not manifest.is_file() or sha_file(manifest) != SOURCE_MANIFEST_SHA256:\n        raise ValueError("This is not the exact 864-record prepared dataset reviewed in this chat.")\n    candidates = [root / "protocol_lock.json", root.parent / "protocol_lock.json"]\n    locks = [p for p in candidates if p.is_file()]\n    if not locks or not any(json.loads(p.read_text()).get("assignment_sha256") == SOURCE_ASSIGNMENT_SHA256 for p in locks):\n        raise ValueError("The source preparation split lock does not match the reviewed version.")\n    rows = list(csv.DictReader(manifest.open()))\n    kept, omitted = filter_complete_families(rows)\n    digest = hashlib.sha256(json.dumps([r["uid"] for r in kept], separators=(",", ":")).encode()).hexdigest()\n    if digest != COHORT_UID_SHA256 or split_counts(kept) != EXPECTED:\n        raise ValueError("The complete-family cohort differs from the pre-training review decision.")\n    for key in ["source_id", "group_id", "rgb_sha256", "phash"]:\n        values = defaultdict(set)\n        for row in kept:\n            values[row[key]].add(row["split"])\n        if any(len(splits) > 1 for splits in values.values()):\n            raise ValueError("Cross-split relationship found: " + key)\n    return kept, omitted\n\n\ndef locate_prepared(input_root, working_root, explicit=None):\n    if explicit:\n        root = Path(explicit)\n        read_preparation(root)\n        return root\n    candidates = []\n    roots = [Path(input_root), Path("/kaggle/working/real_abshar_phase2_preparation_v1")]\n    for parent in roots:\n        if parent.exists():\n            for manifest in parent.rglob("manifest.csv"):\n                root = manifest.parent\n                if (root / "images").is_dir() and (root / "labels").is_dir() and (root / "segmentation_masks").is_dir():\n                    try:\n                        read_preparation(root)\n                        candidates.append(root)\n                    except ValueError:\n                        pass\n    candidates = sorted(set(p.resolve() for p in candidates))\n    if candidates:\n        if len(candidates) > 1:\n            raise ValueError("Multiple reviewed prepared folders are attached. Set PREPARED_ROOT explicitly: " + str(candidates))\n        return candidates[0]\n    archives = sorted(p for p in Path(input_root).rglob("*.zip")\n                      if p.name.startswith("Real-Abshar_Phase2_Prepared_Data"))\n    matching = []\n    for path in archives:\n        with zipfile.ZipFile(path) as archive:\n            if "manifest.csv" in archive.namelist() and hashlib.sha256(archive.read("manifest.csv")).hexdigest() == SOURCE_MANIFEST_SHA256:\n                matching.append(path)\n    if len(matching) == 1:\n        destination = Path(working_root) / "unpacked_preparation"\n        destination.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(matching[0]) as archive:\n            for item in archive.infolist():\n                if item.is_dir():\n                    continue\n                if stat.S_ISLNK(item.external_attr >> 16):\n                    raise ValueError("Dataset archives must not contain symbolic links.")\n                target = safe_path(destination, item.filename)\n                if target.suffix.lower() not in {".png", ".txt", ".csv", ".json", ".yaml"}:\n                    raise ValueError("Unexpected file in the prepared-data archive: " + item.filename)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(item) as source, target.open("wb") as output:\n                    shutil.copyfileobj(source, output)\n        read_preparation(destination)\n        return destination\n    raise FileNotFoundError(\n        "Attach Real-Abshar_Phase2_Prepared_Data.zip as a PRIVATE Kaggle input, or attach the saved "\n        "preparation notebook outputs. The small review ZIP and the original CSI download are not the prepared dataset.")\n\n\ndef verify_row(root, row):\n    with Image.open(safe_path(root, row["export_image"])) as opened:\n        image = opened.convert("RGB")\n        image.load()\n    size = (int(row["width"]), int(row["height"]))\n    if image.size != size or hashlib.sha256(str(size).encode() + image.tobytes()).hexdigest() != row["rgb_sha256"]:\n        raise ValueError("Image integrity check failed: " + row["uid"])\n    with Image.open(safe_path(root, row["export_mask"])) as opened:\n        mask = np.asarray(opened, dtype=np.uint8)\n    if mask.shape != (size[1], size[0]) or not set(np.unique(mask)).issubset({0, 1}):\n        raise ValueError("Invalid 0/1 segmentation mask: " + row["uid"])\n    if hashlib.sha256(str(size).encode() + mask.tobytes()).hexdigest() != row["mask_sha256"]:\n        raise ValueError("Mask integrity check failed: " + row["uid"])\n    line = safe_path(root, row["export_detection_label"]).read_text().strip()\n    expected_box = json.loads(row["box_xyxy_exclusive"])\n    if row["kind"] == "pristine":\n        if line or mask.any() or expected_box is not None:\n            raise ValueError("Pristine image has a positive target: " + row["uid"])\n    else:\n        yy, xx = np.where(mask > 0)\n        box = [int(xx.min()), int(yy.min()), int(xx.max()) + 1, int(yy.max()) + 1]\n        if box != expected_box:\n            raise ValueError("Mask and box disagree: " + row["uid"])\n        values = list(map(float, line.split()))\n        x0, y0, x1, y1 = box\n        expected = [(x0 + x1) / (2 * size[0]), (y0 + y1) / (2 * size[1]),\n                    (x1 - x0) / size[0], (y1 - y0) / size[1]]\n        if len(values) != 5 or values[0] != 0 or not np.allclose(values[1:], expected, atol=1e-8):\n            raise ValueError("Incorrect YOLO label: " + row["uid"])\n\n\ndef build_training_view(prepared_root, work_root):\n    prepared_root, work_root = Path(prepared_root), Path(work_root)\n    rows, omitted = read_preparation(prepared_root)\n    cohort_dir = work_root / "cohort"\n    cohort_dir.mkdir(parents=True, exist_ok=True)\n    lock = {"cohort": COHORT_NAME, "source_manifest_sha256": SOURCE_MANIFEST_SHA256,\n            "source_assignment_sha256": SOURCE_ASSIGNMENT_SHA256, "cohort_uid_sha256": COHORT_UID_SHA256,\n            "rule": "retain only one pristine plus all five QC-valid edits per source; preserve existing split",\n            "split_counts": EXPECTED, "selected_before_any_model_training": True}\n    lock_path = cohort_dir / "cohort_lock.json"\n    if lock_path.exists() and json.loads(lock_path.read_text()) != lock:\n        raise ValueError("A different cohort is already locked in this M4 work folder.")\n    dump(lock_path, lock)\n    write_csv(cohort_dir / "manifest.csv", rows)\n    write_csv(cohort_dir / "valid_records_omitted_by_cohort_rule.csv", omitted)\n    for split in ["train", "val", "test"]:\n        write_csv(cohort_dir / f"{split}_manifest.csv", [r for r in rows if r["split"] == split])\n    view = work_root / "data_view"\n    fit_rows = [r for r in rows if r["split"] in {"train", "val"}]\n    for index, row in enumerate(fit_rows, 1):\n        verify_row(prepared_root, row)\n        for name in ["export_image", "export_detection_label"]:\n            target = safe_path(view, row[name])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(safe_path(prepared_root, row[name]), target)\n        if index % 100 == 0 or index == len(fit_rows):\n            print(f"Verified/copied train+val {index}/{len(fit_rows)}")\n    data_yaml = view / "data.yaml"\n    data_yaml.write_text("path: " + json.dumps(str(view.resolve()))\n                         + "\\ntrain: images/train\\nval: images/val\\nnames:\\n  0: edited_region\\n")\n    dump(cohort_dir / "preflight.json", {"train_val_records_verified": len(fit_rows),\n         "test_image_or_mask_pixels_read": False, "test_images_copied_to_fit_view": False,\n         "original_preparation_records": 864, "cohort_records": len(rows),\n         "valid_records_omitted_by_complete_family_rule": len(omitted), "split_counts": EXPECTED})\n    return rows, data_yaml\n', 'segmentation_data.py': '"""Exact image/mask pairing, letterbox geometry and train/validation-only exports."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\n\nimport numpy as np\nfrom PIL import Image\n\nfrom cohort_data import (COHORT_NAME, COHORT_UID_SHA256, EXPECTED, SOURCE_MANIFEST_SHA256,\n                         SOURCE_ASSIGNMENT_SHA256, dump, read_preparation, safe_path,\n                         verify_row, write_csv)\n\nINPUT_SIZE = 640\nMEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)\nSTD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)\n\n\ndef geometry(width, height, size=INPUT_SIZE):\n    scale = size / max(width, height)\n    w = min(size, max(1, int(round(width * scale))))\n    h = min(size, max(1, int(round(height * scale))))\n    return {"width": w, "height": h, "left": (size - w) // 2, "top": (size - h) // 2, "size": size}\n\n\ndef transform(image, mask, flip=False, size=INPUT_SIZE):\n    image = image.convert("RGB")\n    mask = np.asarray(mask, dtype=np.uint8)\n    if mask.shape != (image.height, image.width) or not set(np.unique(mask)).issubset({0, 1}):\n        raise ValueError("Expected an aligned native 0/1 mask.")\n    if flip:\n        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        mask = np.fliplr(mask).copy()\n    g = geometry(image.width, image.height, size)\n    rgb = np.zeros((size, size, 3), np.uint8)\n    target = np.zeros((size, size), np.uint8)\n    valid = np.zeros((size, size), np.uint8)\n    sy, sx = slice(g["top"], g["top"] + g["height"]), slice(g["left"], g["left"] + g["width"])\n    rgb[sy, sx] = np.asarray(image.resize((g["width"], g["height"]), Image.Resampling.BILINEAR))\n    target[sy, sx] = np.asarray(Image.fromarray(mask).resize((g["width"], g["height"]), Image.Resampling.NEAREST))\n    valid[sy, sx] = 1\n    normalized = (rgb.astype(np.float32).transpose(2, 0, 1) / 255.0 - MEAN) / STD\n    return normalized.copy(), target[None].astype(np.float32), valid[None].astype(np.float32)\n\n\ndef restore_probability(probability, width, height, size=INPUT_SIZE):\n    probability = np.asarray(probability, dtype=np.float32)\n    if probability.shape != (size, size) or not np.isfinite(probability).all():\n        raise ValueError("Expected finite square probability map.")\n    g = geometry(width, height, size)\n    cropped = probability[g["top"]:g["top"] + g["height"], g["left"]:g["left"] + g["width"]]\n    restored = np.asarray(Image.fromarray(cropped).resize((width, height), Image.Resampling.BILINEAR), dtype=np.float32)\n    return np.clip(restored, 0.0, 1.0)\n\n\ndef flip_for(uid, epoch, seed=42):\n    digest = hashlib.sha256(f"{seed}:{epoch}:{uid}".encode()).digest()\n    return bool(digest[0] & 1)\n\n\ndef prepare_view(prepared_root, work_root):\n    root, work = Path(prepared_root), Path(work_root)\n    rows, omitted = read_preparation(root)\n    cohort = work / "cohort"\n    lock = {"cohort": COHORT_NAME, "cohort_uid_sha256": COHORT_UID_SHA256,\n            "source_manifest_sha256": SOURCE_MANIFEST_SHA256,\n            "source_assignment_sha256": SOURCE_ASSIGNMENT_SHA256, "split_counts": EXPECTED,\n            "source_assignments_same_as_M4": True}\n    lock_path = cohort / "cohort_lock.json"\n    if lock_path.exists() and json.loads(lock_path.read_text()) != lock:\n        raise ValueError("Different cohort already locked in this M5 output folder.")\n    dump(lock_path, lock)\n    write_csv(cohort / "manifest.csv", rows)\n    write_csv(cohort / "valid_records_omitted_by_cohort_rule.csv", omitted)\n    for split in ["train", "val", "test"]:\n        write_csv(cohort / f"{split}_manifest.csv", [r for r in rows if r["split"] == split])\n    fit = [r for r in rows if r["split"] in {"train", "val"}]\n    rasterization = []\n    for index, row in enumerate(fit, 1):\n        verify_row(root, row)\n        for key in ["export_image", "export_mask"]:\n            source, target = safe_path(root, row[key]), safe_path(work / "data_view", row[key])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(source, target)\n        with Image.open(safe_path(root, row["export_mask"])) as opened:\n            mask = np.asarray(opened, dtype=np.uint8)\n        g = geometry(int(row["width"]), int(row["height"]))\n        resized = np.asarray(Image.fromarray(mask).resize((g["width"], g["height"]), Image.Resampling.NEAREST))\n        rasterization.append({"uid": row["uid"], "split": row["split"], "kind": row["kind"],\n                              "native_foreground_pixels": int(mask.sum()), "resized_foreground_pixels": int(resized.sum()),\n                              "foreground_lost_in_nearest_resize": bool(mask.any() and not resized.any())})\n        if index % 100 == 0 or index == len(fit):\n            print(f"Verified/copied train+val masks/images: {index}/{len(fit)}", flush=True)\n    write_csv(cohort / "mask_rasterization_audit.csv", rasterization)\n    dump(cohort / "preflight.json", {"verified_train_val_records": len(fit), "split_counts": EXPECTED,\n         "network_input": [INPUT_SIZE, INPUT_SIZE], "test_pixels_decoded_for_M5": False,\n         "test_inference_performed": False,\n         "positive_masks_lost_in_nearest_resize": sum(r["foreground_lost_in_nearest_resize"] for r in rasterization),\n         "scoring": "restore probabilities to native dimensions; compare with untouched native 0/1 masks"})\n    return rows\n', 'segmentation_metrics.py': '"""Native-pixel binary segmentation metrics with explicit empty-mask handling."""\nimport math\nimport numpy as np\n\nTHRESHOLDS = [round(i / 20, 2) for i in range(1, 20)]\n\n\ndef counts(truth, probability, threshold):\n    truth = np.asarray(truth)\n    probability = np.asarray(probability)\n    if truth.shape != probability.shape or truth.ndim != 2:\n        raise ValueError("Native truth/probability map dimensions must match.")\n    if not set(np.unique(truth)).issubset({0, 1}) or not np.isfinite(probability).all():\n        raise ValueError("Expected binary truth and finite probabilities.")\n    if probability.min() < 0 or probability.max() > 1:\n        raise ValueError("Probabilities are outside [0,1].")\n    gt, pred = truth.astype(bool), probability >= threshold\n    tp = int(np.count_nonzero(gt & pred))\n    fp = int(np.count_nonzero(~gt & pred))\n    fn = int(np.count_nonzero(gt & ~pred))\n    tn = int(gt.size - tp - fp - fn)\n    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}\n\n\ndef from_counts(tp, fp, fn, tn):\n    foreground_union = tp + fp + fn\n    background_union = tn + fp + fn\n    iou = tp / foreground_union if foreground_union else None\n    background_iou = tn / background_union if background_union else None\n    dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None\n    background_dice = 2 * tn / (2 * tn + fp + fn) if 2 * tn + fp + fn else None\n    return {"foreground_iou": iou, "foreground_dice": dice,\n            "background_iou": background_iou, "mean_iou_2_classes": (iou + background_iou) / 2\n            if iou is not None and background_iou is not None else None,\n            "background_dice": background_dice,\n            "mean_dice_2_classes": (dice + background_dice) / 2 if dice is not None and background_dice is not None else None,\n            "pixel_precision": tp / (tp + fp) if tp + fp else 0.0,\n            "pixel_recall": tp / (tp + fn) if tp + fn else 0.0,\n            "pixel_accuracy": (tp + tn) / (tp + fp + fn + tn) if tp + fp + fn + tn else None}\n\n\ndef summarize(records):\n    total = {key: sum(int(r[key]) for r in records) for key in ["tp", "fp", "fn", "tn"]}\n    positive = [r for r in records if r["kind"] == "edited"]\n    negative = [r for r in records if r["kind"] == "pristine"]\n    result = {**total, **from_counts(**total), "images": len(records),\n              "edited_images": len(positive), "pristine_images": len(negative)}\n    for key, metric in [("positive_image_mean_dice", "foreground_dice"), ("positive_image_mean_iou", "foreground_iou")]:\n        values = [from_counts(**{k: int(r[k]) for k in total})[metric] for r in positive]\n        result[key] = float(np.mean(values)) if values else None\n    result["pristine_images_with_any_false_foreground"] = sum(int(r["fp"]) > 0 for r in negative)\n    result["pristine_any_pixel_false_alarm_rate"] = (result["pristine_images_with_any_false_foreground"] / len(negative)\n                                                      if negative else None)\n    result["pristine_mean_false_foreground_fraction"] = (float(np.mean([int(r["fp"]) / (int(r["fp"]) + int(r["tn"]))\n                                                                        for r in negative])) if negative else None)\n    area_errors = [abs(int(r["fp"]) - int(r["fn"])) / sum(int(r[k]) for k in total) for r in records]\n    positive_area_errors = [abs(int(r["fp"]) - int(r["fn"])) / (int(r["tp"]) + int(r["fn"])) for r in positive]\n    result["mean_absolute_area_fraction_error"] = float(np.mean(area_errors)) if area_errors else None\n    result["edited_median_relative_area_error"] = float(np.median(positive_area_errors)) if positive_area_errors else None\n    return result\n\n\ndef choose_threshold(grid):\n    return max(grid, key=lambda r: (r["foreground_iou"], r["foreground_dice"], r["pixel_precision"], r["threshold"]))\n\n\ndef size_slices(records):\n    result = {}\n    for name, low, high in [("small", 0, 32 ** 2), ("medium", 32 ** 2, 96 ** 2), ("large", 96 ** 2, float("inf"))]:\n        group = [r for r in records if r["kind"] == "edited" and low <= float(r["mask_area_at_input"]) < high]\n        result[name] = summarize(group) if group else {"images": 0}\n    return result\n', 'segmentation_models.py': '"""FCN-style and U-Net-style binary segmenters sharing a ResNet18 encoder."""\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\nfrom torchvision.models import resnet18\n\nfrom cohort_data import safe_path\nfrom segmentation_data import transform, flip_for\n\n\nclass SegmentationDataset(Dataset):\n    def __init__(self, root, rows, training=False, epoch=0):\n        self.root, self.rows, self.training, self.epoch = root, rows, training, epoch\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        with Image.open(safe_path(self.root, row["export_image"])) as opened:\n            image = opened.convert("RGB")\n        with Image.open(safe_path(self.root, row["export_mask"])) as opened:\n            mask = np.asarray(opened, dtype=np.uint8)\n        image, mask, valid = transform(image, mask, flip=self.training and flip_for(row["uid"], self.epoch))\n        return {"image": torch.from_numpy(image), "mask": torch.from_numpy(mask),\n                "valid": torch.from_numpy(valid), "index": index}\n\n\nclass Encoder(nn.Module):\n    def __init__(self, pretrained=None):\n        super().__init__()\n        backbone = resnet18(weights=None)\n        if pretrained is not None:\n            backbone.load_state_dict(torch.load(pretrained, map_location="cpu", weights_only=True))\n        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)\n        self.pool = backbone.maxpool\n        self.layer1, self.layer2 = backbone.layer1, backbone.layer2\n        self.layer3, self.layer4 = backbone.layer3, backbone.layer4\n\n    def forward(self, x):\n        x0 = self.stem(x)\n        x1 = self.layer1(self.pool(x0))\n        x2 = self.layer2(x1)\n        x3 = self.layer3(x2)\n        x4 = self.layer4(x3)\n        return x0, x1, x2, x3, x4\n\n\nclass Block(nn.Sequential):\n    def __init__(self, inputs, outputs):\n        super().__init__(nn.Conv2d(inputs, outputs, 3, padding=1, bias=False),\n                         nn.GroupNorm(8, outputs), nn.ReLU(inplace=True),\n                         nn.Conv2d(outputs, outputs, 3, padding=1, bias=False),\n                         nn.GroupNorm(8, outputs), nn.ReLU(inplace=True))\n\n\nclass FCNSegmenter(nn.Module):\n    def __init__(self, pretrained=None):\n        super().__init__()\n        self.encoder = Encoder(pretrained)\n        self.head = nn.Sequential(Block(512, 128), nn.Dropout2d(0.1), nn.Conv2d(128, 1, 1))\n\n    def forward(self, x):\n        logits = self.head(self.encoder(x)[-1])\n        return F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)\n\n\nclass UNetSegmenter(nn.Module):\n    def __init__(self, pretrained=None):\n        super().__init__()\n        self.encoder = Encoder(pretrained)\n        self.blocks = nn.ModuleList([Block(512 + 256, 128), Block(128 + 128, 64),\n                                    Block(64 + 64, 32), Block(32 + 64, 16)])\n        self.final = nn.Sequential(Block(16, 8), nn.Conv2d(8, 1, 1))\n\n    def forward(self, x):\n        x0, x1, x2, x3, current = self.encoder(x)\n        for block, skip in zip(self.blocks, [x3, x2, x1, x0], strict=True):\n            current = F.interpolate(current, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n            current = block(torch.cat([current, skip], dim=1))\n        current = F.interpolate(current, size=x.shape[-2:], mode="bilinear", align_corners=False)\n        return self.final(current)\n\n\ndef build_model(architecture, pretrained=None):\n    if architecture == "A_FCN_ResNet18":\n        return FCNSegmenter(pretrained)\n    if architecture == "B_UNet_ResNet18":\n        return UNetSegmenter(pretrained)\n    raise ValueError("Unknown segmentation architecture: " + architecture)\n\n\ndef segmentation_loss(logits, target, valid):\n    """Mean valid-pixel BCE per image + soft Dice on positive resized masks; padding has zero gradient."""\n    axes = (1, 2, 3)\n    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")\n    bce = ((bce * valid).sum(axes) / valid.sum(axes).clamp_min(1)).mean()\n    probability = logits.sigmoid() * valid\n    truth = target * valid\n    positives = truth.sum(axes) > 0\n    dice = (2 * (probability * truth).sum(axes) + 1e-6) / (probability.sum(axes) + truth.sum(axes) + 1e-6)\n    dice_loss = (1 - dice[positives]).mean() if positives.any() else logits.sum() * 0\n    return 0.5 * bce + 0.5 * dice_loss\n', 'segmentation_report.py': '"""Report figures with split, input resolution and threshold context on every page."""\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom cohort_data import safe_path\nfrom segmentation_metrics import from_counts\n\n\ndef make_figures(work, rows, comparisons, grid, selection, per_image):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    work = Path(work)\n    out = work / "figures"\n    out.mkdir(parents=True, exist_ok=True)\n    context = "Validation • letterbox 640×640 • native-pixel scoring • no postprocessing"\n    selected_threshold = selection["probability_threshold"]\n    with (work / "metrics" / "val_before_finetuning_comparison.csv").open() as stream:\n        before = {r["architecture"]: float(r["foreground_iou"]) for r in csv.DictReader(stream)}\n    fig, ax = plt.subplots(figsize=(11, 5.5))\n    x = np.arange(len(comparisons))\n    ax.bar(x - 0.27, [before[r["architecture"]] for r in comparisons], 0.18, label="Before fine-tuning: IoU at 0.50", color="#a8a8a8")\n    ax.bar(x - 0.09, [r["fixed_0.5_foreground_iou"] for r in comparisons], 0.18, label="Fine-tuned: IoU at 0.50")\n    ax.bar(x + 0.09, [r["foreground_iou"] for r in comparisons], 0.18, label="Fine-tuned: IoU at tuned threshold")\n    ax.bar(x + 0.27, [r["foreground_dice"] for r in comparisons], 0.18, label="Fine-tuned: Dice at tuned threshold")\n    ax.set_xticks(x, [r["architecture"] + f\'\\nt ≥ {r["threshold"]:.2f}\' for r in comparisons], fontsize=9)\n    ax.set(ylim=(0, 1), ylabel="Global foreground score")\n    ax.set_title(context + "\\nBefore/fixed: t = 0.50; tuned thresholds shown under model names", fontsize=10)\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n    fig.savefig(out / "val_segmenter_comparison.png", dpi=160)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(9, 5))\n    for name in [r["architecture"] for r in comparisons]:\n        values = [r for r in grid if r["architecture"] == name]\n        ax.plot([r["threshold"] for r in values], [r["foreground_iou"] for r in values], marker=".", label=name)\n    metric = selection["validation_metrics"]\n    ax.scatter([selected_threshold], [metric["foreground_iou"]], marker="*", s=140, color="black", zorder=5, label="Selected")\n    ax.set(xlabel="Probability threshold (0.05…0.95, step 0.05)", ylabel="Global foreground IoU", ylim=(0, 1))\n    ax.set_title(context + "\\nValidation threshold sweep; foreground if probability ≥ threshold", fontsize=10)\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out / "val_segmentation_threshold_sweep.png", dpi=160)\n    plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    for row in comparisons:\n        with (work / "runs" / row["architecture"] / "results.csv").open() as stream:\n            history = list(csv.DictReader(stream))\n        epochs = [int(r["epoch"]) for r in history]\n        short = row["architecture"].split("_")[1]\n        axes[0].plot(epochs, [float(r["train_loss"]) for r in history], label=short + " train")\n        axes[0].plot(epochs, [float(r["val_loss"]) for r in history], linestyle="--", label=short + " val")\n        axes[1].plot(epochs, [float(r["val_foreground_iou_at_0.5"]) for r in history], label=short)\n    axes[0].set(xlabel="Epoch", ylabel="Masked BCE + positive-mask soft Dice loss")\n    axes[1].set(xlabel="Epoch", ylabel="Validation foreground IoU at t = 0.50", ylim=(0, 1))\n    for ax in axes:\n        ax.legend(fontsize=8)\n    fig.suptitle("Train/validation • letterbox 640×640\\nLoss excludes padding; IoU uses native masks at threshold 0.50; no postprocessing", fontsize=10)\n    fig.tight_layout()\n    fig.savefig(out / "train_val_segmentation_learning_curves.png", dpi=160)\n    plt.close(fig)\n\n    matrix = np.array([[metric["tn"], metric["fp"]], [metric["fn"], metric["tp"]]], dtype=np.int64)\n    normalized = matrix / np.maximum(matrix.sum(1, keepdims=True), 1)\n    fig, ax = plt.subplots(figsize=(7, 5.5))\n    ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")\n    for i in range(2):\n        for j in range(2):\n            ax.text(j, i, f"{matrix[i,j]:,}\\n{normalized[i,j]:.1%}", ha="center", va="center",\n                    color="white" if normalized[i,j] > 0.5 else "black", fontsize=10)\n    ax.set_xticks([0, 1], ["Background", "Edited region"])\n    ax.set_yticks([0, 1], ["Background", "Edited region"])\n    ax.set(xlabel="Predicted pixel class", ylabel="True pixel class")\n    ax.set_title(context + f"\\nThreshold ≥ {selected_threshold:.2f}; counts and percentages within true-class rows", fontsize=9)\n    fig.tight_layout()\n    fig.savefig(out / "val_pixel_confusion_matrix.png", dpi=160)\n    plt.close(fig)\n\n    # Six deterministic development examples; the final test failure gallery is still reserved.\n    positives = [r for r in per_image if r["kind"] == "edited"]\n    negatives = [r for r in per_image if r["kind"] == "pristine"]\n    dice = lambda r: from_counts(**{k: int(r[k]) for k in ["tp", "fp", "fn", "tn"]})["foreground_dice"]\n    candidates = [r["uid"] for r in sorted(positives, key=lambda r: (dice(r), r["uid"]))[:2]]\n    candidates += [r["uid"] for r in sorted(positives, key=lambda r: (-dice(r), r["uid"]))[:2]]\n    candidates += [r["uid"] for r in sorted(negatives, key=lambda r: (-r["fp"], r["uid"]))[:1]]\n    candidates += [r["uid"] for r in sorted(negatives, key=lambda r: (r["fp"], r["uid"]))[:1]]\n    uids = list(dict.fromkeys(candidates))\n    uids += [r["uid"] for r in per_image if r["uid"] not in uids]\n    uids = uids[:6]\n    by_uid = {r["uid"]: r for r in rows}\n    fig, axes = plt.subplots(len(uids), 4, figsize=(14, 3 * len(uids)), squeeze=False)\n    for line, uid in enumerate(uids):\n        row = by_uid[uid]\n        with Image.open(safe_path(work / "data_view", row["export_image"])) as opened:\n            rgb = np.asarray(opened.convert("RGB"))\n        with Image.open(safe_path(work / "data_view", row["export_mask"])) as opened:\n            truth = np.asarray(opened, dtype=np.uint8)\n        with Image.open(work / "val_prediction_masks" / (uid + ".png")) as opened:\n            predicted = np.asarray(opened, dtype=np.uint8)\n        overlay = rgb.astype(np.float32) / 255\n        overlay[predicted > 0] = 0.55 * overlay[predicted > 0] + 0.45 * np.array([1.0, 0.15, 0.1])\n        for ax, value, title in zip(axes[line], [rgb, truth, predicted, overlay],\n                                    [uid, "Native ground truth", "Predicted 0/1 mask", "Prediction overlay"], strict=True):\n            ax.imshow(value, cmap="gray" if value.ndim == 2 else None, vmin=0 if value.ndim == 2 else None,\n                      vmax=1 if value.ndim == 2 else None)\n            ax.set_title(title, fontsize=8)\n            ax.axis("off")\n    fig.suptitle(context + f"\\n{selection[\'selected_architecture\']} • probability ≥ {selected_threshold:.2f}\\n"\n                 "Validation development examples; white/red = edited region", fontsize=11)\n    fig.tight_layout(rect=(0, 0, 1, 0.95))\n    fig.savefig(out / "val_segmentation_development_gallery.png", dpi=120)\n    plt.close(fig)\n', 'm5_backup.py': '"""Back up model checkpoints and experiment metadata, without dataset images/masks."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nimport shutil\nimport tempfile\nimport uuid\n\nfrom cohort_data import dump, safe_path, sha_file\n\nBACKUP_EVERY_EPOCHS = 5\nBACKUP_WAIT_SECONDS = 300\n\n\ndef backup_files(work_root):\n    root = Path(work_root)\n    selected = set()\n    for name in ["m5_protocol_lock.json", "environment.json", "m5_selection.json", "backup_session_id.txt"]:\n        path = root / name\n        if path.is_file():\n            selected.add(path)\n    for folder, suffixes in {"cohort": {".csv", ".json"}, "metrics": {".csv", ".json"},\n                             "validation_predictions": {".json"}, "wandb_ids": {".txt", ".json"},\n                             "source": {".py"}}.items():\n        for path in (root / folder).glob("*"):\n            if path.is_file() and path.suffix.lower() in suffixes:\n                selected.add(path)\n    pretrained = root / "pretrained" / "resnet18_imagenet.pth"\n    if pretrained.is_file():\n        selected.add(pretrained)\n    for run in ["A_FCN_ResNet18", "B_UNet_ResNet18"]:\n        for relative in ["args.yaml", "results.csv", "training_complete.json", "weights/best.pt", "weights/last.pt"]:\n            path = root / "runs" / run / relative\n            if path.is_file():\n                selected.add(path)\n    # Scientific plots are regenerated from backed-up numbers. No photographs enter the artifact.\n    for path in selected:\n        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):\n            raise ValueError("Backup file escapes its work folder: " + str(path))\n    return sorted(selected)\n\n\ndef should_backup(epoch, stopping=False):\n    return epoch == 1 or epoch % BACKUP_EVERY_EPOCHS == 0 or stopping\n\n\ndef upload_backup(run, work_root, stage):\n    import wandb\n    if not os.environ.get("WANDB_API_KEY") or getattr(run.settings, "mode", None) != "online":\n        raise RuntimeError("Online W&B is required for recovery backups. Enable the WANDB_API_KEY secret and Internet before training.")\n    root = Path(work_root)\n    session = root / "backup_session_id.txt"\n    if not session.exists():\n        session.write_text(uuid.uuid4().hex[:12])\n    paths = backup_files(root)\n    if not paths:\n        raise RuntimeError("No experiment files exist to back up.")\n    manifest = {"schema": "real-abshar-m5-backup-v1", "stage": stage,\n                "created_utc": datetime.now(timezone.utc).isoformat(),\n                "dataset_images_or_masks_included": False,\n                "files": {p.relative_to(root).as_posix(): {"sha256": sha_file(p), "bytes": p.stat().st_size}\n                          for p in paths}}\n    manifest_path = root / "cloud_backup_manifest.json"\n    dump(manifest_path, manifest)\n    artifact = wandb.Artifact("real-abshar-m5-backup-" + session.read_text().strip(), type="model",\n                              metadata={"stage": stage, "dataset_images_included": False,\n                                        "file_count": len(paths), "test_used": False})\n    for path in paths:\n        artifact.add_file(str(path), name=path.relative_to(root).as_posix(), policy="mutable")\n    artifact.add_file(str(manifest_path), name="cloud_backup_manifest.json", policy="mutable")\n    logged = run.log_artifact(artifact, aliases=["latest"])\n    logged.wait(timeout=BACKUP_WAIT_SECONDS)\n    # Publish a receipt only after the SDK confirms the upload completed.\n    receipt = {"stage": stage, "artifact": logged.qualified_name, "url": logged.url,\n               "confirmed_utc": datetime.now(timezone.utc).isoformat(),\n               "file_count": len(paths), "dataset_images_or_masks_included": False}\n    dump(root / "cloud_backup_receipt.json", receipt)\n    run.summary["latest_model_backup"] = receipt["artifact"]\n    print("BACKUP_CONFIRMED:", stage, receipt["artifact"], flush=True)\n    print("Backup link:", receipt["url"], flush=True)\n    return receipt\n\n\ndef restore_backup(reference, destination):\n    """Restore an own-project artifact into its original work path; reject corrupt/conflicting files."""\n    import wandb\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix="m5_restore_", dir=destination.parent) as directory:\n        downloaded = Path(wandb.Api().artifact(reference).download(root=directory))\n        manifest = json.loads((downloaded / "cloud_backup_manifest.json").read_text())\n        if manifest.get("schema") != "real-abshar-m5-backup-v1" or manifest.get("dataset_images_or_masks_included") is not False:\n            raise ValueError("Unexpected model-backup schema.")\n        for name, expected in manifest["files"].items():\n            source, target = safe_path(downloaded, name), safe_path(destination, name)\n            if not source.is_file() or source.stat().st_size != expected["bytes"] or sha_file(source) != expected["sha256"]:\n                raise ValueError("Corrupt backup file: " + name)\n            if target.exists() and (not target.is_file() or sha_file(target) != expected["sha256"]):\n                raise ValueError("Restore would replace a different local file: " + str(target))\n        for name in manifest["files"]:\n            source, target = safe_path(downloaded, name), safe_path(destination, name)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(source, target)\n        shutil.copyfile(downloaded / "cloud_backup_manifest.json", destination / "cloud_backup_manifest.json")\n    print("Restored model/metadata backup:", reference)\n    print("Keep the original WORK_ROOT path and attach the prepared dataset before resuming.")\n    return destination\n', 'm5_train.py': '"""Train two edited-region segmenters, select on validation, preserve test for final integration."""\nfrom pathlib import Path\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport time\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nfrom cohort_data import COHORT_NAME, COHORT_UID_SHA256, EXPECTED, dump, locate_prepared, safe_path, sha_file, write_csv\nfrom segmentation_data import INPUT_SIZE, prepare_view, restore_probability\nfrom segmentation_metrics import THRESHOLDS, counts, summarize, choose_threshold, size_slices\nfrom m5_backup import upload_backup, should_backup\n\nARCHITECTURES = ["A_FCN_ResNet18", "B_UNet_ResNet18"]\nSETTINGS = {"input_size": INPUT_SIZE, "batch": 4, "epochs": 25, "patience": 6, "seed": 42,\n            "workers": 2, "precision": "FP32", "optimizer": "AdamW", "learning_rate": 0.0003,\n            "weight_decay": 0.0001, "scheduler": "CosineAnnealingLR", "minimum_lr": 0.000001,\n            "gradient_clip_norm": 1.0, "horizontal_flip_probability": 0.5,\n            "loss": "0.5 valid-pixel BCE per image + 0.5 soft Dice on positive resized masks",\n            "checkpoint_selection": "highest native-pixel validation foreground IoU at threshold 0.5",\n            "threshold_selection": "highest native-pixel validation foreground IoU; tie Dice, precision, higher threshold",\n            "model_selection": "highest tuned validation foreground IoU; tie Dice, precision, higher threshold, then A",\n            "postprocessing": "none", "normalization_mean": [0.485, 0.456, 0.406],\n            "normalization_std": [0.229, 0.224, 0.225]}\nPRETRAINED_URL = "https://download.pytorch.org/models/resnet18-f37072fd.pth"\n\n\ndef seed(value):\n    import torch\n    random.seed(value)\n    np.random.seed(value)\n    torch.manual_seed(value)\n    torch.cuda.manual_seed_all(value)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    torch.set_float32_matmul_precision("highest")\n\n\ndef cleanup():\n    import torch\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef start_wandb(work, project, entity, name, configuration):\n    import wandb\n    if not os.environ.get("WANDB_API_KEY"):\n        raise RuntimeError("Enable the WANDB_API_KEY Kaggle secret; online checkpoint backup is required.")\n    path = work / "wandb_ids" / (name + ".txt")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        path.write_text(wandb.util.generate_id())\n    run = wandb.init(project=project, entity=entity, name=name, id=path.read_text().strip(), resume="allow",\n                     mode="online", dir=str(work), save_code=False, config=configuration,\n                     tags=["phase2", "M5", COHORT_NAME])\n    dump(path.with_suffix(".json"), {"project": project, "entity": run.entity, "id": run.id, "url": run.url, "mode": "online"})\n    return run\n\n\ndef preflight(work):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU accelerator before running M5.")\n    packages = {}\n    for name in ["torch", "torchvision", "wandb", "numpy", "Pillow", "matplotlib"]:\n        packages[name] = importlib.metadata.version(name)\n    if packages["wandb"] != "0.28.1":\n        raise RuntimeError("Run the pinned W&B dependency cell in a fresh kernel.")\n    environment = {"python": platform.python_version(), "packages": packages,\n                   "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}\n    pretrained = work / "pretrained" / "resnet18_imagenet.pth"\n    pretrained.parent.mkdir(parents=True, exist_ok=True)\n    if not pretrained.exists():\n        temp = pretrained.with_suffix(".download")\n        torch.hub.download_url_to_file(PRETRAINED_URL, str(temp), hash_prefix="f37072fd", progress=True)\n        temp.replace(pretrained)\n    if not sha_file(pretrained).startswith("f37072fd"):\n        raise ValueError("The ImageNet pretrained checkpoint checksum prefix is wrong.")\n    protocol = {"cohort": COHORT_NAME, "cohort_uid_sha256": COHORT_UID_SHA256, "split_counts": EXPECTED,\n                "architectures": ARCHITECTURES, "settings": SETTINGS, "threshold_grid": THRESHOLDS,\n                "pretrained_url": PRETRAINED_URL, "pretrained_sha256": sha_file(pretrained),\n                "source_sha256": {p.name: sha_file(p) for p in Path(__file__).parent.glob("*.py")},\n                "environment": environment, "test_evaluation_performed": False,\n                "scoring": "crop letterbox padding; bilinear probability resize to native size; untouched native GT",\n                "empty_mask_convention": "empty/empty foreground IoU and Dice undefined; aggregate confusion and negatives separately",\n                "backup": "online W&B before training, epoch 1, every 5 epochs, stopping, completed runs and final selection"}\n    protocol["before_finetuning_baseline"] = "Each architecture with ImageNet encoder/random decoder, seed 42, no task training; val threshold 0.50"\n    lock = work / "m5_protocol_lock.json"\n    if lock.exists() and json.loads(lock.read_text()) != protocol:\n        raise ValueError("M5 protocol/environment changed in this work folder. Restore its original environment/code or use a new folder.")\n    dump(lock, protocol)\n    dump(work / "environment.json", environment)\n    return pretrained, protocol\n\n\ndef smoke_checks(work, pretrained):\n    import torch\n    from segmentation_models import build_model, segmentation_loss\n    device = torch.device("cuda:0")\n    checks = {}\n    for architecture in ARCHITECTURES:\n        seed(42)\n        model = build_model(architecture, pretrained).to(device).eval()\n        image = torch.zeros((2, 3, 96, 128), device=device)\n        truth = torch.zeros((2, 1, 96, 128), device=device)\n        truth[0, :, 24:64, 32:96] = 1\n        valid = torch.ones_like(truth)\n        valid[:, :, :8] = 0\n        logits = model(image)\n        if logits.shape != truth.shape:\n            raise ValueError("Segmentation model returned the wrong output dimensions.")\n        loss = segmentation_loss(logits, truth, valid)\n        if not torch.isfinite(loss):\n            raise ValueError("Segmentation smoke-test loss is not finite.")\n        loss.backward()\n        gradients = [p.grad for p in model.parameters() if p.grad is not None]\n        if not gradients or not all(torch.isfinite(g).all() for g in gradients) or not any(g.abs().sum() > 0 for g in gradients):\n            raise ValueError("No finite nonzero model gradient in smoke test.")\n        checks[architecture] = {"output_shape": list(logits.shape), "finite_loss_and_gradients": True,\n                                "parameter_count": sum(p.numel() for p in model.parameters())}\n        del model, logits, loss\n        cleanup()\n    logits = torch.zeros((2, 1, 8, 8), device=device, requires_grad=True)\n    target = torch.zeros_like(logits)\n    target[0, :, 2:6, 2:6] = 1\n    valid = torch.zeros_like(logits)\n    valid[:, :, 1:7, 1:7] = 1\n    loss = segmentation_loss(logits, target, valid)\n    loss.backward()\n    if torch.count_nonzero(logits.grad[valid == 0]) != 0:\n        raise ValueError("Padded pixels affected the training gradient.")\n    original_loss = float(loss.detach())\n    with torch.no_grad():\n        logits[valid == 0] = 100\n    if abs(float(segmentation_loss(logits, target, valid).detach()) - original_loss) > 1e-6:\n        raise ValueError("Changing padding logits changed the loss.")\n    checks["padding_gradient_is_zero"] = True\n    negative_logits = torch.zeros((2, 1, 8, 8), device=device, requires_grad=True)\n    negative_loss = segmentation_loss(negative_logits, torch.zeros_like(negative_logits), torch.ones_like(negative_logits))\n    negative_loss.backward()\n    if not torch.isfinite(negative_loss) or not torch.isfinite(negative_logits.grad).all() or not (negative_logits.grad > 0).all():\n        raise ValueError("All-pristine batch does not have the expected finite background-suppressing gradient.")\n    checks["all_pristine_batch_loss_and_gradient"] = True\n    dump(work / "metrics" / "gpu_smoke_checks.json", checks)\n    print("GPU model shape/gradient and padding checks passed.", flush=True)\n\n\ndef loader(work, rows, training=False, epoch=0):\n    import torch\n    from torch.utils.data import DataLoader\n    from segmentation_models import SegmentationDataset\n    dataset = SegmentationDataset(work / "data_view", rows, training=training, epoch=epoch)\n    generator = torch.Generator().manual_seed(SETTINGS["seed"] + 1009 * epoch)\n    return DataLoader(dataset, batch_size=SETTINGS["batch"], shuffle=training, num_workers=SETTINGS["workers"],\n                      pin_memory=True, generator=generator, persistent_workers=False)\n\n\ndef evaluate(model, work, rows, thresholds=(0.5,), save_predictions=None):\n    import torch\n    from segmentation_models import segmentation_loss\n    model.eval()\n    records = {float(t): [] for t in thresholds}\n    total_loss = 0.0\n    if save_predictions:\n        Path(save_predictions).mkdir(parents=True, exist_ok=True)\n    with torch.inference_mode():\n        for batch in loader(work, rows):\n            image, target, valid = [batch[key].to("cuda:0", non_blocking=True) for key in ["image", "mask", "valid"]]\n            logits = model(image)\n            loss = segmentation_loss(logits, target, valid)\n            if not torch.isfinite(loss):\n                raise ValueError("Non-finite validation loss.")\n            total_loss += float(loss) * image.shape[0]\n            probabilities = logits.sigmoid().cpu().numpy()[:, 0]\n            for index, probability in zip(batch["index"].tolist(), probabilities, strict=True):\n                row = rows[index]\n                width, height = int(row["width"]), int(row["height"])\n                native = restore_probability(probability, width, height)\n                with Image.open(safe_path(work / "data_view", row["export_mask"])) as opened:\n                    truth = np.asarray(opened, dtype=np.uint8)\n                common = {"uid": row["uid"], "source_id": row["source_id"], "kind": row["kind"],\n                          "split": "val", "manipulation": row["manipulation"],\n                          "mask_area_at_input": float(truth.sum()) * (INPUT_SIZE / max(width, height)) ** 2}\n                for threshold in thresholds:\n                    records[float(threshold)].append({**common, "threshold": float(threshold), **counts(truth, native, threshold)})\n                if save_predictions:\n                    Image.fromarray((native >= thresholds[0]).astype(np.uint8)).save(Path(save_predictions) / (row["uid"] + ".png"))\n    grid = [{"threshold": threshold, **summarize(values)} for threshold, values in records.items()]\n    return grid, records, total_loss / len(rows)\n\n\ndef atomic_save(value, path):\n    import torch\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".partial")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef model_fingerprint(model):\n    digest = hashlib.sha256()\n    for name, tensor in model.state_dict().items():\n        value = tensor.detach().cpu().contiguous()\n        digest.update(name.encode())\n        digest.update(str(tuple(value.shape)).encode())\n        digest.update(str(value.dtype).encode())\n        digest.update(value.numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef before_finetuning_baselines(work, rows, pretrained):\n    from segmentation_models import build_model\n    values = []\n    protocol_sha = sha_file(work / "m5_protocol_lock.json")\n    for architecture in ARCHITECTURES:\n        path = work / "metrics" / (architecture + "_before_finetuning.json")\n        if path.exists():\n            record = json.loads(path.read_text())\n            if record["protocol_sha256"] != protocol_sha:\n                raise ValueError("Before-fine-tuning baseline belongs to a different protocol.")\n        else:\n            seed(SETTINGS["seed"])\n            model = build_model(architecture, pretrained).to("cuda:0").eval()\n            fingerprint = model_fingerprint(model)\n            grid, records, loss = evaluate(model, work, rows)\n            record = {"architecture": architecture, "stage": "before_finetuning", "task_training_epochs": 0,\n                      "split": "val", "input_size": INPUT_SIZE, "threshold": 0.5,\n                      "initial_parameter_sha256": fingerprint, "protocol_sha256": protocol_sha,\n                      "validation_metrics": grid[0], "validation_loss": loss,\n                      "initialization": "ImageNet encoder, random task decoder; not a pretrained edit-mask predictor"}\n            dump(path, record)\n            write_csv(work / "metrics" / (architecture + "_before_finetuning_per_image.csv"), records[0.5])\n            print("Before-fine-tuning validation baseline:", architecture, grid[0]["foreground_iou"], flush=True)\n            del model\n            cleanup()\n        values.append({"architecture": architecture, "stage": "before_finetuning", "split": "val", "input_size": INPUT_SIZE,\n                       **record["validation_metrics"]})\n    write_csv(work / "metrics" / "val_before_finetuning_comparison.csv", values)\n\n\ndef train_architecture(architecture, work, train_rows, val_rows, pretrained, project, entity):\n    import torch\n    from segmentation_models import build_model, segmentation_loss\n    directory = work / "runs" / architecture\n    weights = directory / "weights"\n    best_path, last_path = weights / "best.pt", weights / "last.pt"\n    marker = directory / "training_complete.json"\n    if marker.exists():\n        metadata = json.loads(marker.read_text())\n        if best_path.is_file() and sha_file(best_path) == metadata["best_sha256"]:\n            print("Reusing completed M5 training:", architecture, flush=True)\n            return best_path\n        raise ValueError("Completed segmentation checkpoint changed: " + architecture)\n    if (directory / "results.csv").exists() and not last_path.exists():\n        raise ValueError("Partial M5 run has no last.pt; restore its W&B backup rather than overwriting it.")\n    directory.mkdir(parents=True, exist_ok=True)\n    # The backup allowlist stores args.yaml, so keep a JSON-compatible YAML representation.\n    dump(directory / "args.yaml", {"architecture": architecture, **SETTINGS})\n    seed(SETTINGS["seed"])\n    model = build_model(architecture, pretrained).to("cuda:0")\n    initial = json.loads((work / "metrics" / (architecture + "_before_finetuning.json")).read_text())\n    if model_fingerprint(model) != initial["initial_parameter_sha256"]:\n        raise ValueError("The training initialization differs from its before-fine-tuning baseline.")\n    optimizer = torch.optim.AdamW(model.parameters(), lr=SETTINGS["learning_rate"], weight_decay=SETTINGS["weight_decay"])\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS["epochs"], eta_min=SETTINGS["minimum_lr"])\n    epoch_start, best_score, best_epoch, bad_epochs, history = 0, -1.0, 0, 0, []\n    protocol_sha = sha_file(work / "m5_protocol_lock.json")\n    if last_path.exists():\n        checkpoint = torch.load(last_path, map_location="cpu", weights_only=True)\n        if checkpoint["protocol_sha256"] != protocol_sha or not best_path.is_file() or sha_file(best_path) != checkpoint["best_sha256"]:\n            raise ValueError("Resume checkpoint/protocol/best.pt mismatch; restore the last confirmed W&B snapshot.")\n        model.load_state_dict(checkpoint["model_state"])\n        optimizer.load_state_dict(checkpoint["optimizer_state"])\n        scheduler.load_state_dict(checkpoint["scheduler_state"])\n        epoch_start, best_score = checkpoint["epoch"], checkpoint["best_score"]\n        best_epoch, bad_epochs, history = checkpoint["best_epoch"], checkpoint["bad_epochs"], checkpoint["history"]\n        write_csv(directory / "results.csv", history)\n        del checkpoint\n        print("Resuming", architecture, "after epoch", epoch_start, flush=True)\n    run = start_wandb(work, project, entity, architecture, {"cohort": COHORT_NAME, "architecture": architecture, **SETTINGS})\n    started = time.monotonic()\n    try:\n        upload_backup(run, work, architecture + "_preflight")\n        for epoch in range(epoch_start, SETTINGS["epochs"]):\n            if bad_epochs >= SETTINGS["patience"]:\n                break\n            seed(SETTINGS["seed"] + 1009 * (epoch + 1))\n            model.train()\n            total_loss = 0.0\n            for batch in loader(work, train_rows, training=True, epoch=epoch + 1):\n                image, target, valid = [batch[key].to("cuda:0", non_blocking=True) for key in ["image", "mask", "valid"]]\n                optimizer.zero_grad(set_to_none=True)\n                logits = model(image)\n                loss = segmentation_loss(logits, target, valid)\n                if not torch.isfinite(loss):\n                    raise ValueError("Non-finite training loss.")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), SETTINGS["gradient_clip_norm"], error_if_nonfinite=True)\n                optimizer.step()\n                total_loss += float(loss.detach()) * image.shape[0]\n            grid, _, val_loss = evaluate(model, work, val_rows)\n            metric = grid[0]\n            score = metric["foreground_iou"]\n            improved = score > best_score\n            if improved:\n                best_score, best_epoch, bad_epochs = score, epoch + 1, 0\n                atomic_save({"architecture": architecture, "epoch": best_epoch, "model_state": model.state_dict(),\n                             "protocol_sha256": protocol_sha, "validation_at_0.5": metric}, best_path)\n            else:\n                bad_epochs += 1\n            scheduler.step()\n            history.append({"epoch": epoch + 1, "train_loss": total_loss / len(train_rows), "val_loss": val_loss,\n                            "val_foreground_iou_at_0.5": score, "val_foreground_dice_at_0.5": metric["foreground_dice"],\n                            "val_mean_iou_2_classes_at_0.5": metric["mean_iou_2_classes"], "learning_rate_next": optimizer.param_groups[0]["lr"]})\n            write_csv(directory / "results.csv", history)\n            atomic_save({"architecture": architecture, "epoch": epoch + 1, "model_state": model.state_dict(),\n                         "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),\n                         "best_score": best_score, "best_epoch": best_epoch, "best_sha256": sha_file(best_path),\n                         "bad_epochs": bad_epochs, "history": history, "protocol_sha256": protocol_sha}, last_path)\n            run.log(history[-1])\n            print(f\'{architecture} epoch {epoch+1}/{SETTINGS["epochs"]}: train loss={history[-1]["train_loss"]:.4f}, \'\n                  f\'val foreground IoU@0.5={score:.4f}, best epoch={best_epoch}\', flush=True)\n            stop = bad_epochs >= SETTINGS["patience"] or epoch + 1 == SETTINGS["epochs"]\n            if should_backup(epoch + 1, stop):\n                upload_backup(run, work, f"{architecture}_epoch_{epoch+1}")\n        metadata = {"architecture": architecture, "completed_epochs": len(history), "best_epoch": best_epoch,\n                    "best_validation_foreground_iou_at_0.5": best_score, "best_sha256": sha_file(best_path),\n                    "session_training_seconds": time.monotonic() - started, "test_evaluated": False}\n        dump(marker, metadata)\n        run.summary.update(metadata)\n        upload_backup(run, work, architecture + "_complete")\n    finally:\n        run.finish()\n        del model, optimizer, scheduler\n        cleanup()\n    return best_path\n\n\ndef load_trained(checkpoint):\n    import torch\n    from segmentation_models import build_model\n    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    model = build_model(saved["architecture"])\n    model.load_state_dict(saved["model_state"])\n    return model.to("cuda:0").eval()\n\n\ndef package(work, selection):\n    review = work.parent / "Real-Abshar_Phase2_M5_Results_Review.zip"\n    models = work.parent / "Real-Abshar_Phase2_M5_Models.zip"\n    with zipfile.ZipFile(review, "w", zipfile.ZIP_DEFLATED) as archive:\n        for folder in ["cohort", "metrics", "figures", "source", "wandb_ids"]:\n            for path in sorted((work / folder).rglob("*")):\n                if path.is_file() and "__pycache__" not in path.parts:\n                    archive.write(path, path.relative_to(work).as_posix())\n        for name in ["m5_protocol_lock.json", "m5_selection.json", "environment.json", "cloud_backup_receipt.json"]:\n            archive.write(work / name, name)\n        for architecture in ARCHITECTURES:\n            for name in ["results.csv", "args.yaml", "training_complete.json"]:\n                path = work / "runs" / architecture / name\n                archive.write(path, path.relative_to(work).as_posix())\n    with zipfile.ZipFile(models, "w", zipfile.ZIP_DEFLATED) as archive:\n        for architecture in ARCHITECTURES:\n            archive.write(work / "runs" / architecture / "weights" / "best.pt", f"models/{architecture}.pt")\n        archive.write(work / "pretrained" / "resnet18_imagenet.pth", "pretrained/resnet18_imagenet.pth")\n        for name in ["m5_protocol_lock.json", "m5_selection.json", "environment.json", "cloud_backup_receipt.json"]:\n            archive.write(work / name, name)\n        for path in (work / "source").glob("*.py"):\n            archive.write(path, "source/" + path.name)\n    dump(work / "output_archives.json", {"review": {"name": review.name, "sha256": sha_file(review)},\n         "models": {"name": models.name, "sha256": sha_file(models)}, "test_evaluation_performed": False})\n    return review, models\n\n\ndef run(input_root="/kaggle/input", work_root="/kaggle/working/real_abshar_phase2_m5_v1", prepared_root=None,\n        wandb_project="Real-Abshar-Phase2", wandb_entity=None):\n    work = Path(work_root)\n    work.mkdir(parents=True, exist_ok=True)\n    root = locate_prepared(input_root, work, prepared_root)\n    rows = prepare_view(root, work)\n    train_rows, val_rows = [r for r in rows if r["split"] == "train"], [r for r in rows if r["split"] == "val"]\n    pretrained, protocol = preflight(work)\n    smoke_checks(work, pretrained)\n    before_finetuning_baselines(work, val_rows, pretrained)\n    checkpoints, comparisons, all_grid = {}, [], []\n    for architecture in ARCHITECTURES:\n        checkpoint = train_architecture(architecture, work, train_rows, val_rows, pretrained, wandb_project, wandb_entity)\n        checkpoints[architecture] = checkpoint\n        model = load_trained(checkpoint)\n        grid, records, loss = evaluate(model, work, val_rows, THRESHOLDS)\n        grid = [{"architecture": architecture, "split": "val", "input_size": INPUT_SIZE, **r} for r in grid]\n        all_grid.extend(grid)\n        write_csv(work / "metrics" / (architecture + "_per_image_thresholds.csv"),\n                  [r for threshold in THRESHOLDS for r in records[threshold]])\n        selected = choose_threshold(grid)\n        fixed = next(r for r in grid if r["threshold"] == 0.5)\n        comparisons.append({"architecture": architecture, "checkpoint_sha256": sha_file(checkpoint),\n                            "threshold": selected["threshold"], "foreground_iou": selected["foreground_iou"],\n                            "foreground_dice": selected["foreground_dice"], "pixel_precision": selected["pixel_precision"],\n                            "pixel_recall": selected["pixel_recall"], "mean_iou_2_classes": selected["mean_iou_2_classes"],\n                            "fixed_0.5_foreground_iou": fixed["foreground_iou"],\n                            "fixed_0.5_foreground_dice": fixed["foreground_dice"], "split": "val", "input_size": INPUT_SIZE})\n        del model\n        cleanup()\n    write_csv(work / "metrics" / "val_threshold_sweep.csv", all_grid)\n    write_csv(work / "metrics" / "val_model_comparison.csv", comparisons)\n    selected = choose_threshold(comparisons)\n    architecture, threshold = selected["architecture"], selected["threshold"]\n    model = load_trained(checkpoints[architecture])\n    grid, records, _ = evaluate(model, work, val_rows, [threshold], work / "val_prediction_masks")\n    per_image = records[threshold]\n    write_csv(work / "metrics" / "val_selected_per_image.csv", per_image)\n    dump(work / "metrics" / "val_selected_size_slices.json", size_slices(per_image))\n    selection = {"selected_architecture": architecture, "checkpoint_sha256": sha_file(checkpoints[architecture]),\n                 "model_file_in_archive": f"models/{architecture}.pt", "input_size": [INPUT_SIZE, INPUT_SIZE],\n                 "probability_threshold": threshold, "postprocessing": "none", "selection_split": "val",\n                 "cohort_uid_sha256": COHORT_UID_SHA256, "validation_metrics": grid[0],\n                 "prediction_values": [0, 1], "test_evaluation_performed": False,\n                 "integration_note": "Run segmentation independently of detector presence; do not gate on a detection.",\n                 "scope": "132-family CSI complete aligned subset; binary edit/inpainting regions; not Saudi deployment performance"}\n    dump(work / "m5_selection.json", selection)\n    from segmentation_report import make_figures\n    make_figures(work, val_rows, comparisons, all_grid, selection, per_image)\n    del model\n    cleanup()\n    record = start_wandb(work, wandb_project, wandb_entity, "M5_validation_selection",\n                         {"cohort": COHORT_NAME, "selected_architecture": architecture, "threshold": threshold, "test_used": False})\n    try:\n        record.log({"val/" + k: v for k, v in grid[0].items() if isinstance(v, (int, float))})\n        record.summary.update({"selected_architecture": architecture, "threshold": threshold, "test_evaluated": False})\n        upload_backup(record, work, "M5_completed")\n    finally:\n        record.finish()\n    review, models = package(work, selection)\n    print(json.dumps(selection, indent=2))\n    print("SEND HERE:", review)\n    print("KEEP FOR INTEGRATION:", models)\n    return selection, review, models\n'}
SOURCE_DIR = WORK_ROOT / 'source'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for filename, source in SOURCES.items():
    (SOURCE_DIR / filename).write_text(source)
sys.path.insert(0, str(SOURCE_DIR))
for name in ['m5_train', 'm5_backup', 'segmentation_models', 'segmentation_report', 'segmentation_metrics', 'segmentation_data', 'cohort_data']:
    sys.modules.pop(name, None)
from m5_backup import restore_backup
if RESTORE_ARTIFACT:
    restore_backup(RESTORE_ARTIFACT, WORK_ROOT)
from cohort_data import EXPECTED, read_preparation, locate_prepared
ROOT = locate_prepared(INPUT_ROOT, WORK_ROOT, PREPARED_ROOT)
COHORT, OMITTED = read_preparation(ROOT)
print('Prepared dataset:', ROOT)
print('Frozen cohort:', EXPECTED)
print('Complete-family omissions:', len(OMITTED))
print('Test inference executed: False')


## 4. Train, compare and select using validation

This rechecks the 678 train/validation image/mask targets, audits nearest-mask rasterization,
verifies the official ImageNet weight checksum prefix and checks model shapes/finite gradients,
padding exclusion and all-pristine batch behavior on the GPU. A live W&B artifact upload is
confirmed before fitting. Checkpoints are backed up during and after training.

Loss: 0.5 valid-pixel BCE + 0.5 soft Dice on positive resized masks. Padding contributes no loss.
Native scoring: crop padding, bilinearly resize probabilities to the original image dimensions,
then threshold and compare with the original binary mask. No mask geometry repair or morphology.

Primary model/threshold criterion: validation global foreground IoU. Positive-image Dice and
pristine false-foreground diagnostics are reported separately. The test split stays reserved.


In [ ]:
# 4. Actual GPU training and validation selection. No test evaluation.
from m5_train import run

RESULT, REVIEW_ZIP, MODELS_ZIP = run(
    input_root=INPUT_ROOT, work_root=WORK_ROOT, prepared_root=ROOT,
    wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY)


In [ ]:
# 5. Download both archives NOW, then Quick Save with output files enabled.
from IPython.display import FileLink, Image as DisplayImage, display

display(FileLink(str(REVIEW_ZIP)))
display(FileLink(str(MODELS_ZIP)))
for filename in ['val_segmenter_comparison.png', 'val_segmentation_threshold_sweep.png',
                 'train_val_segmentation_learning_curves.png', 'val_pixel_confusion_matrix.png',
                 'val_segmentation_development_gallery.png']:
    path = WORK_ROOT / 'figures' / filename
    if path.exists():
        display(DisplayImage(filename=str(path)))
print('SEND HERE:', REVIEW_ZIP.name)
print('KEEP FOR INTEGRATION:', MODELS_ZIP.name)
print('Test evaluation performed:', RESULT['test_evaluation_performed'])
print('Cloud backup receipt:')
print((WORK_ROOT / 'cloud_backup_receipt.json').read_text())


## Preserve and return results

Wait for `BACKUP_CONFIRMED: M5_completed`, download both ZIPs and verify a saved notebook version
contains its output files before ending the GPU session. Return the small Results_Review ZIP.
Keep the Models ZIP for integration. Source code and model settings are included in both archives.

The new W&B artifacts back up models and metadata; dataset photographs/masks and photo galleries
are excluded. If restoring this run later, use the same notebook, work path and environment;
set RESTORE_ARTIFACT to your own confirmed M5 reference and attach the prepared-data input.

Next is M8 integration and validation, followed by one final test evaluation after all settings
are frozen. The report still needs the supplied Phase 2 Overleaf template, final code commit,
W&B/service/model-card links and actual final metrics. Your three-person team deletes Section 5.
CPU checks passed; real neural training and cloud access are verified by this Kaggle run.
